# EDA: Fraudulent Job Postings

Exploring the EMSCAD job postings dataset before modeling: class imbalance, missing
data by field, text length, and the words that most distinguish fraudulent postings
from real ones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from collections import Counter
import re

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

df = pd.read_csv("../data/fake_job_postings.csv")
print(df.shape)
df.head(3)

## Class balance

How rare is fraud in this dataset? This directly determines the modeling approach
(class weighting, which metrics to trust).

In [ ]:
fraud_rate = df["fraudulent"].mean()
print(f"Fraudulent: {df['fraudulent'].sum():,} ({fraud_rate:.1%})")
print(f"Real: {(df['fraudulent'] == 0).sum():,} ({1 - fraud_rate:.1%})")

ax = df["fraudulent"].value_counts().sort_index().plot(
    kind="bar", figsize=(4, 4), color=["#4C72B0", "#C44E52"]
)
ax.set_xticklabels(["Real", "Fraudulent"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Class balance")
plt.tight_layout()
plt.show()

**Takeaway:** with roughly a 20:1 imbalance, accuracy is a useless metric here — a model that always predicts "real" would score ~95% accuracy while catching zero fraud. This is why the baseline and BERT scripts both report precision/recall/F1/PR-AUC instead, and why the BERT training uses class-weighted loss.

## Missing data by field

Which text fields are commonly blank, and does that differ between real and fraudulent postings? A field that's suspiciously empty (or suspiciously generic) on fraud postings is itself a signal.

In [ ]:
text_fields = ["title", "company_profile", "description", "requirements", "benefits"]

missing_by_class = df.groupby("fraudulent")[text_fields].apply(
    lambda g: g.isna().mean()
).T
missing_by_class.columns = ["real", "fraudulent"]
missing_by_class.round(3)

In [ ]:
missing_by_class.plot(kind="bar", figsize=(8, 4), color=["#4C72B0", "#C44E52"])
plt.ylabel("Fraction missing")
plt.title("Missing-field rate: real vs. fraudulent postings")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Text length distribution

Do fraudulent postings tend to be shorter (less effort put into a fake listing) or longer (padded with generic-sounding text)?

In [ ]:
def strip_html(text):
    if not isinstance(text, str):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

df["combined_text"] = (
    df[text_fields].fillna("").agg(" ".join, axis=1).apply(strip_html)
)
df["text_length"] = df["combined_text"].str.split().apply(len)

plt.figure(figsize=(8, 4))
sns.histplot(
    data=df, x="text_length", hue="fraudulent", bins=50, stat="density",
    common_norm=False, palette=["#4C72B0", "#C44E52"]
)
plt.xlim(0, 1000)
plt.title("Posting length (words): real vs. fraudulent")
plt.xlabel("Word count")
plt.show()

df.groupby("fraudulent")["text_length"].describe()[["mean", "50%", "min", "max"]]

## Most distinctive words per class

A quick, model-free look at which words appear disproportionately in fraudulent vs. real postings. This previews what the TF-IDF baseline's coefficients and the BERT model's SHAP explanations should pick up on later.

In [ ]:
STOPWORDS = set(
    "the a an and or of to in for on with is are be this that as at by".split()
)

def top_words(texts, n=25):
    counter = Counter()
    for t in texts:
        words = re.findall(r"[a-z']+", t.lower())
        counter.update(w for w in words if w not in STOPWORDS and len(w) > 2)
    return counter.most_common(n)

fraud_words = top_words(df.loc[df["fraudulent"] == 1, "combined_text"])
real_words = top_words(df.loc[df["fraudulent"] == 0, "combined_text"])

comparison = pd.DataFrame({
    "fraudulent_top_words": [w for w, _ in fraud_words],
    "real_top_words": [w for w, _ in real_words],
})
comparison

## Notes for modeling

- Severe class imbalance (~5% fraud) → use class weighting and report PR-AUC/F1, not accuracy.
- `company_profile` and `benefits` are missing far more often on fraudulent postings — missingness itself could be a useful feature.
- Text length and raw word frequency give a first read on what separates the classes, but a lot of the real signal is likely in phrasing/context, which is exactly what fine-tuning BERT (vs. the TF-IDF baseline) should capture better.
- Carry these observations into the README's "what the model picks up on" section once SHAP results are in.